# Capital One — Data Science Assessment (Simulation)

4 questions, CodeSignal-style. Suggested budget: **~70 minutes total** (Q1 10m, Q2 20m, Q3 20m, Q4 20m).

**Rules of the sim**
- Write your solution in the `# YOUR SOLUTION` cell for each question, then run the grader cell below it.
- Each question has its own folder (`q1/`, `q2/`, ...) and you write your output file into that folder.
- Q3 and Q4 have an 8-second execution limit in the real OA — the grader prints your runtime.
- Don't open `.solutions/` — that's the answer key the graders read.

Regenerate fresh data any time with `python generate_data.py`.

In [1]:
%load_ext autoreload
%autoreload 2

import os, time
import numpy as np
import pandas as pd

import grader  # graders resolve paths relative to this folder, so cwd doesn't matter

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)
print("ready")

ready


---
## Question 1 of 4 — Basic data analysis

You are provided with datasets containing information about taxi drivers and their rides. Perform basic
data analysis and save the results to a CSV file.

Data (in `q1/`):

**`drivers.csv`**
- `driver_id` (int) — unique driver identifier
- `age` (int)
- `second_language` (str) — `"no"` if the driver has none
- `rating` (float) — driver's average rating

**`rides_{i}.csv`** — split into 4 files, `rides_1.csv` … `rides_4.csv`
- `ride_id` (int), `driver_id` (int), `passenger_id` (int), `date` (str)
- `status` (str) — one of `["Rejected by the driver", "Cancelled by the passenger", "Success"]`

**Tasks**
1. **Average driver rating** — mean of the `rating` column. Store as `insight_type: "average_driver_rating"`.
2. **Percentage of drivers with a second language** — share of drivers where `second_language != "no"`, as a
   percentage. Store as `insight_type: "percentage_drivers_with_second_language"`.
3. **Ride success rate** — combine all four `rides_{i}.csv` files, then compute the percentage of rides with
   `status == "Success"`. Store as `insight_type: "ride_success_rate"`.

**Output:** save `q1/analysis_results.csv` with two columns, `insight_type` and `value`, one row per task.
Numeric values are correct if they match to two decimal places.

`q1/tests/data_analysis_tests_data/expected.csv` shows the expected output format and the expected value of
`average_driver_rating`. The other two values there are zero placeholders, not the real answers.

In [34]:
# YOUR SOLUTION — Question 1
_t0 = time.time()

# drivers
# rides 
drivers = pd.read_csv('q1/drivers.csv')
average_driver_rating = drivers['rating'].mean()
# print(average_driver_rating)
drivers.dropna(subset='second_language')
print(drivers)

num_non_second = len(drivers[
    drivers['second_language'] != "no"
])
percentage_drivers_with_second_language = (num_non_second)/len(drivers)*100
print(percentage_drivers_with_second_language)



riders = pd.DataFrame()
for i in range(1, 5):
    rider = pd.read_csv(f"./q1/rides_{i}.csv")
    riders = pd.concat([riders, rider])
#     print(len(rider))

num_succ = len(riders[riders['status']=='Success'])
ride_success_rate = (num_succ/len(riders))*100
print(ride_success_rate)
# print(len(riders))
print(riders)

final = pd.DataFrame({
    "insight_type": ["average_driver_rating", "percentage_drivers_with_second_language", "ride_success_rate"],
    "value": [average_driver_rating, percentage_drivers_with_second_language, ride_success_rate]
})

final.to_csv("q1/analysis_results.csv", index=False)

# ... write q1/analysis_results.csv

# print(f"runtime: {time.time() - _t0:.2f}s")

      driver_id  age second_language  rating
0             1   35              no    3.93
1             2   37          Arabic    3.98
2             3   22          Arabic    4.19
3             4   65              no    4.20
4             5   58              no    4.20
...         ...  ...             ...     ...
2995       2996   44         Russian    5.00
2996       2997   31              no    4.95
2997       2998   22              no    4.90
2998       2999   46              no    4.61
2999       3000   39              no    5.00

[3000 rows x 4 columns]
44.5
74.22833333333332
       ride_id  driver_id  passenger_id        date                      status
0        15282       1802         37177  2022-09-15                     Success
1        21436       2412         12480  2022-10-05                     Success
2        44537        468         23882  2023-03-26  Cancelled by the passenger
3        13519        626            75  2023-03-15  Cancelled by the passenger
4        475

In [35]:
grader.grade_q1()

=== Question 1: basic analysis ===
[PASS] analysis_results.csv exists  -> /Users/log/Github/CPT/capital1_oa/q1/analysis_results.csv
[PASS] columns are ['insight_type', 'value']  -> ['insight_type', 'value']
[PASS] average_driver_rating == 4.42  -> got 4.4238
[PASS] percentage_drivers_with_second_language == 44.50  -> got 44.5000
[PASS] ride_success_rate == 74.23  -> got 74.2283
--- 5/5 checks passed ---


True

---
## Question 2 of 4 — Feature collection

Data about taxi drivers and their rides, created by **April 15th, 2023**. When calculating any time
features, treat **April 15th, 2023 as today**.

Data (in `q2/`), across 6 files:

**`drivers.csv`** — `driver_id` (int), `car_id` (int), `age` (int), `started_driving_year` (int),
`second_language` (str, `"no"` if none), `rating` (float), `net_worth_of_tips` (float),
`driver_class` (str: `"A class"` / `"B class"`)

**`rides_{i}.csv`** (4 files) — `ride_id`, `driver_id`, `passenger_id`, `date`, `status`,
`car_clearness_upvote_given` (bool), `politeness_upvote_given` (bool), `communication_upvote_given` (bool),
`punctuality_upvote_given` (bool), `complaint_given` (bool)

**`cars.csv`** — `car_id` (int), `model` (str), `manufacture_year` (int), `last_inspection_date` (str)

**Task:** retrieve the needed information about each driver and store it in **`q2/collected.csv`** with columns:

| column | type | notes |
|---|---|---|
| `driver_id` | int | |
| `car_model` | str | |
| `car_manufacture_year` | int | |
| `days_since_inspection` | int | days passed since the last inspection |
| `age` | int | |
| `experience` | int | `2023 - started_driving_year` |
| `second_language` | str | |
| `rating` | float | |
| `net_worth_of_tips` | float | |
| `number_of_upvotes` | int | total upvotes across all four upvote flags, over all of the driver's rides |
| `driver_class` | str | |

Rows and columns may be in any order — the tests are order-agnostic.

In [86]:
# YOUR SOLUTION — Question 2
_t0 = time.time()

drivers = pd.read_csv("q2/drivers.csv")
cars = pd.read_csv("q2/cars.csv")
rides = pd.concat([pd.read_csv(f"q2/rides_{i}.csv") for i in range(1, 5)], ignore_index=True)
TODAY = pd.Timestamp("2023-04-15")

merged = drivers.merge(
    cars,
    left_on="car_id",
    right_on="car_id"
)

merged = merged.rename(columns={
    "model": "car_model",
    "manufacture_year": "car_manufacture_year"
})

merged['days_since_inspection'] = (
    TODAY - pd.to_datetime(merged['last_inspection_date'])).dt.days

merged['experience'] = (
    TODAY.year - merged['started_driving_year']
)

driver_rides = merged.merge(
    rides,
    on="driver_id"
)
# merged['net_worth_of_tips'] = (
#     rides['']
# )
# print(rides)
# merged['number_of_upvotes'] =

# print(merged.groupby('driver_id')[
#     (merged['car_clearness_upvote_given']) & 
#     (merged['politeness_upvote_given']) &
#     (merged['communication_upvote_given']) &
#     (merged['punctuality_upvote_given'])
# ])
# print(merged.groupby('driver_id')['car_clearness_upvote_given']==True)
driver_rides['number_of_upvotes'] = (driver_rides['car_clearness_upvote_given'].astype(int) 
                               + driver_rides['politeness_upvote_given'].astype(int)
                               + driver_rides['communication_upvote_given'].astype(int)
                               + driver_rides['punctuality_upvote_given'].astype(int))
# print(driver_rides)
upvotes = driver_rides.groupby('driver_id')['number_of_upvotes'].sum().reset_index()
merged = merged.merge(
    upvotes,
    on='driver_id'
)
# print(merged)


cols = ['driver_id', 'car_model', 'car_manufacture_year', 'days_since_inspection', 'age', 'experience', 'second_language', 'rating', 'net_worth_of_tips', 'number_of_upvotes', 'driver_class']
final = merged[cols]
print(final)
final.to_csv('q2/collected.csv')
# ... write q2/collected.csv

print(f"runtime: {time.time() - _t0:.2f}s")

      driver_id         car_model  car_manufacture_year  days_since_inspection  age  experience second_language  rating  net_worth_of_tips  number_of_upvotes driver_class
0             1     Nissan Altima                  2022                    351   35           2              no    3.93             384.11                 19      B class
1             2       Ford Fusion                  2015                     15   37          17          Arabic    3.98             537.09                 25      B class
2             3        Kia Optima                  2011                    264   22           2          Arabic    4.19             635.68                 16      B class
3             4      Toyota Camry                  2017                    671   65           9              no    4.20             335.35                 20      B class
4             5      Honda Accord                  2020                    651   58          16              no    4.20             609.44       

In [87]:
grader.grade_q2()

=== Question 2: feature collection ===
[PASS] collected.csv exists  -> /Users/log/Github/CPT/capital1_oa/q2/collected.csv
[PASS] all required columns present  -> missing []
[PASS] row count == 3000  -> got 3000
[PASS] driver_id set matches
[PASS] car_model matches  -> 0 mismatched rows
[PASS] car_manufacture_year matches  -> 0 mismatched rows
[PASS] days_since_inspection matches  -> 0 mismatched rows
[PASS] age matches  -> 0 mismatched rows
[PASS] experience matches  -> 0 mismatched rows
[PASS] second_language matches  -> 0 mismatched rows
[PASS] rating matches  -> 0 mismatched rows
[PASS] net_worth_of_tips matches  -> 0 mismatched rows
[PASS] number_of_upvotes matches  -> 0 mismatched rows
[PASS] driver_class matches  -> 0 mismatched rows
--- 14/14 checks passed ---


True

---
## Question 3 of 4 — Data preparation

A dataset of taxi drivers and their performance metrics, with columns:

`driver_id` (int), `car_model` (str), `car_manufacture_year` (int), `days_since_inspection` (int),
`age` (int), `experience` (int), `second_language` (str), `rating` (float), `net_worth_of_tips` (float),
`number_of_rejected_rides` (int), `number_of_upvotes` (int), `number_of_complaints` (int),
`number_of_incidents` (int), `driver_class` (str)

Split: **train 70%** at `q3/data/train.csv`, **test 30%** at `q3/data/test.csv`.

**Steps**
- **a.** Fill missing values in `age` with the mean age of the drivers, rounded to the nearest integer.
- **b.** Convert `second_language` and `car_model` to numbers with **ordinal encoding** — codes must start at
  0 and be consecutive integers.
  ✅ `{"Honda Accord": 0, "Ford Fusion": 1, "Hyundai Sonata": 2, "Nissan Altima": 3}`
  ❌ `{"Hyundai Sonata": 1, "Honda Accord": 2, "Ford Fusion": 4, "Nissan Altima": 5}`
- **c.** Normalize `net_worth_of_tips` with **Standard Scaling**.
- **d.** Convert `driver_class`: `"A class"` → 0, `"B class"` → 1.

⚠️ **Do not leak information from the test set into the train set** — fit every statistic (mean age, encoder,
scaler) on train only.

**Output:** `q3/processed_train.csv` and `q3/processed_test.csv`.
Values in `net_worth_of_tips` must be written with **exactly 5 digits after the decimal point**.

**Constraints:** 8 s, 4 GB.

In [ ]:
# YOUR SOLUTION — Question 3
_t0 = time.time()

train = pd.read_csv("q3/data/train.csv")
test = pd.read_csv("q3/data/test.csv")

mean_age = int(round(train['age'].mean(),0))
# print(mean_age)
train['age'] = train['age'].fillna(mean_age)
test['age'] = test['age'].fillna(mean_age)

# print(train)
import sklearn.preprocessing
encoder = sklearn.preprocessing.OrdinalEncoder()
train[['second_language', 'car_model']] = encoder.fit_transform(
    train[['second_language', 'car_model']]
)
test[['second_language', 'car_model']] = encoder.transform(
    test[['second_language', 'car_model']]
)


scaler = sklearn.preprocessing.StandardScaler()
train[['net_worth_of_tips']] = scaler.fit_transform(
    train[['net_worth_of_tips']]
)

test[['net_worth_of_tips']] = scaler.transform(
    test[['net_worth_of_tips']]
)


train['driver_class'] = train['driver_class'].replace({
    "A class": 0,
    "B class": 1})
test['driver_class'] = test['driver_class'].replace({
    "A class": 0,
    "B class": 1})

train['net_worth_of_tips'] = train['net_worth_of_tips'].map(
    lambda x: f"{x:.5f}"
)

test['net_worth_of_tips'] = test['net_worth_of_tips'].map(
    lambda x: f"{x:.5f}"
)

train.to_csv('q3/processed_train.csv')
test.to_csv('q3/processed_test.csv')
# ... write q3/processed_train.csv and q3/processed_test.csv
# hint: df.to_csv(path, index=False, float_format=...) formats every float column --
#       you probably want to format only net_worth_of_tips.

# print(f"runtime: {time.time() - _t0:.2f}s  (limit 8s)")

/var/folders/kp/vdv61pd97vd0x29b257r8h7h0000gn/T/ipykernel_30310/2120441545.py:33: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  train['driver_class'] = train['driver_class'].replace({
/var/folders/kp/vdv61pd97vd0x29b257r8h7h0000gn/T/ipykernel_30310/2120441545.py:36: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  test['driver_class'] = test['driver_class'].replace({


In [126]:
grader.grade_q3()

=== Question 3: preprocessing ===
[PASS] processed_train.csv exists
[PASS] processed_test.csv exists
[PASS] train row count preserved  -> 2100 vs 2100
[PASS] test row count preserved  -> 900 vs 900
[PASS] age has no missing values
[PASS] missing train ages filled with rounded train mean (44)  -> got [44.]
[PASS] missing test ages filled with TRAIN mean (44, no leakage)  -> got [44.]
[PASS] second_language is numeric
[PASS] second_language encoded as 0..6  -> got [0, 1, 2, 3, 4, 5, 6]
[PASS] second_language test codes consistent with train  -> got [0, 1, 2, 3, 4, 5, 6]
[PASS] car_model is numeric
[PASS] car_model encoded as 0..6  -> got [0, 1, 2, 3, 4, 5, 6]
[PASS] car_model test codes consistent with train  -> got [0, 1, 2, 3, 4, 5, 6]
[PASS] train tips standard-scaled  -> mean=-0.0000 std=1.0000
[PASS] test tips scaled with TRAIN stats (no leakage)  -> mean=-0.0047 (leak-free target -0.0047)
[PASS] train tips written with exactly 5 decimals  -> 0 bad rows e.g. -1.01631
[PASS] test tip

True

---
## Question 4 of 4 — Classifier

Using the dataset from the prior question, train a classifier that predicts whether a driver is
**A class (0)** or **B class (1)**.

Free-form task — any model, any libraries.

**Data (in `q4/`)**
- Training set 70% — `q4/data/train.csv`
- Validation set 15% — `q4/data/val.csv`
- Test set 15% — `q4/data/test.csv` (no `driver_class` column)

**Metrics:** precision and recall, with **B class as the positive class**.
**Goal:** maximize recall while keeping precision relatively high.

**Output:** `q4/predictions.csv`, one column named `driver_class`, one row per test row, in test-set order:

```
driver_class
0
1
0
...
```

Scoring in the real OA shows only the first 10 rows immediately; you submit to see the full score. The
grader below mimics that: it prints the first-10 preview and then the full precision/recall.

**Constraints:** 8 s, 4 GB.

In [160]:
# YOUR SOLUTION — Question 4
_t0 = time.time()

train = pd.read_csv("q4/data/train.csv")
val = pd.read_csv("q4/data/val.csv")
test = pd.read_csv("q4/data/test.csv")

mean_age = int(round(train['age'].mean(),0))
# print(mean_age)
train['age'] = train['age'].fillna(mean_age)
test['age'] = test['age'].fillna(mean_age)
val['age'] = val['age'].fillna(mean_age)

# print(train)
import sklearn.preprocessing
encoder = sklearn.preprocessing.OrdinalEncoder()
train[['second_language', 'car_model']] = encoder.fit_transform(
    train[['second_language', 'car_model']]
)
val[['second_language', 'car_model']] = encoder.transform(
    val[['second_language', 'car_model']]
)
test[['second_language', 'car_model']] = encoder.transform(
    test[['second_language', 'car_model']]
)


scaler = sklearn.preprocessing.StandardScaler()
train[['net_worth_of_tips']] = scaler.fit_transform(
    train[['net_worth_of_tips']]
)
val[['net_worth_of_tips']] = scaler.transform(
    val[['net_worth_of_tips']]
)
test[['net_worth_of_tips']] = scaler.transform(
    test[['net_worth_of_tips']]
)


train['driver_class'] = train['driver_class'].replace({
    "A class": 0,
    "B class": 1})
val['driver_class'] = val['driver_class'].replace({
    "A class": 0,
    "B class": 1})


#################33

X_train = train.drop(columns='driver_class')
Y_train = train['driver_class']

X_val = val.drop(columns='driver_class')
Y_val = val['driver_class']


from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score

model = sklearn.linear_model.LogisticRegression()
model.fit(X_train, Y_train)
# pred = model.predict(X_val)
probs = model.predict_proba(X_val)[:, 1]

for threshold in [0.2, 0.3, 0.35, 0.4, 0.5, 0.6, 0.7]:
    pred = (probs >= threshold).astype(int)
    print(threshold)
    
    print(f"precision: {precision_score(Y_val, pred)}")
    print(f"recall: {recall_score(Y_val, pred)}")

# print(train)

test_probs = model.predict_proba(test)[:,1]
test_pred = (test_probs >= 0.45).astype(int)

print(test_pred)

predictions = pd.DataFrame({
    "driver_class": test_pred
})
predictions.to_csv('q4/predictions.csv', index=False)
# ... write q4/predictions.csv

print(f"runtime: {time.time() - _t0:.2f}s  (limit 8s)")

0.2
precision: 0.717948717948718
recall: 0.9781659388646288
0.3
precision: 0.7526501766784452
recall: 0.9301310043668122
0.35
precision: 0.7672727272727272
recall: 0.9213973799126638
0.4
precision: 0.79296875
recall: 0.8864628820960698
0.5
precision: 0.8151260504201681
recall: 0.8471615720524017
0.6
precision: 0.8511627906976744
recall: 0.7991266375545851
0.7
precision: 0.8918918918918919
recall: 0.7205240174672489
[0 0 0 0 0 1 1 1 1 0 1 0 1 1 0 1 1 0 1 0 1 1 0 1 1 1 0 1 0 1 1 1 1 0 0 1 0
 1 0 0 1 1 0 0 0 1 0 1 1 0 0 0 0 1 1 0 0 0 0 0 1 0 1 1 0 1 0 0 0 0 1 0 0 0
 0 0 1 0 1 1 0 1 1 1 0 0 0 0 1 1 0 1 0 0 1 1 1 1 1 1 1 0 0 1 0 1 0 0 1 0 1
 1 0 0 1 0 0 1 0 1 0 0 0 0 0 1 0 0 1 1 1 0 1 1 0 0 1 0 1 1 1 0 1 1 1 0 1 1
 0 0 0 0 1 0 0 1 0 1 0 1 1 1 0 1 0 0 1 1 1 0 0 1 1 1 1 0 1 1 0 1 1 1 1 0 0
 1 0 0 0 1 0 1 0 0 1 0 1 1 0 0 0 0 0 0 1 0 1 0 1 1 1 1 0 1 1 1 1 1 1 0 0 1
 0 1 0 1 0 1 1 0 0 0 1 0 1 0 0 0 0 0 0 0 0 1 0 1 0 0 1 1 0 0 0 1 0 1 1 1 0
 1 1 0 0 0 1 0 1 1 1 1 1 0 1 0 1 0 1 0 0 1 0 0 0 0 0 1 1

/var/folders/kp/vdv61pd97vd0x29b257r8h7h0000gn/T/ipykernel_30310/251394443.py:40: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  train['driver_class'] = train['driver_class'].replace({
/var/folders/kp/vdv61pd97vd0x29b257r8h7h0000gn/T/ipykernel_30310/251394443.py:43: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  val['driver_class'] = val['driver_class'].replace({
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=

In [161]:
grader.grade_q4()


First-10-rows preview score (accuracy): 0.80
Full test set  ->  precision: 0.8515   recall: 0.8667   f1: 0.8590
=== Question 4: classifier ===
[PASS] predictions.csv exists
[PASS] column named 'driver_class'  -> ['driver_class']
[PASS] row count == 450  -> got 450
[PASS] all predictions in {0, 1}
[PASS] recall >= 0.85 (target)  -> 0.8667
[PASS] precision >= 0.80 (target)  -> 0.8515
--- 6/6 checks passed ---


True

---
### Run everything

Re-runs all four graders for a final tally.

In [162]:
for fn in (grader.grade_q1, grader.grade_q2, grader.grade_q3, grader.grade_q4):
    fn()
    print()

=== Question 1: basic analysis ===
[PASS] analysis_results.csv exists  -> /Users/log/Github/CPT/capital1_oa/q1/analysis_results.csv
[PASS] columns are ['insight_type', 'value']  -> ['insight_type', 'value']
[PASS] average_driver_rating == 4.42  -> got 4.4238
[PASS] percentage_drivers_with_second_language == 44.50  -> got 44.5000
[PASS] ride_success_rate == 74.23  -> got 74.2283
--- 5/5 checks passed ---

=== Question 2: feature collection ===
[PASS] collected.csv exists  -> /Users/log/Github/CPT/capital1_oa/q2/collected.csv
[PASS] all required columns present  -> missing []
[PASS] row count == 3000  -> got 3000
[PASS] driver_id set matches
[PASS] car_model matches  -> 0 mismatched rows
[PASS] car_manufacture_year matches  -> 0 mismatched rows
[PASS] days_since_inspection matches  -> 0 mismatched rows
[PASS] age matches  -> 0 mismatched rows
[PASS] experience matches  -> 0 mismatched rows
[PASS] second_language matches  -> 0 mismatched rows
[PASS] rating matches  -> 0 mismatched rows
[P